# Create embeddings of documents

#### Setup Environment

In [23]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
from semantic_text_splitter import TextSplitter
from tokenizers import Tokenizer
import pickle

In [24]:
# Loads variables from the environment
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")

#### Read in and Chunk Documents 

#### Create Embedding Vectors for Chunks

> ToDo: add some sort of loop through all videos

In [25]:
# Load title dictionary from pickle file
with open('data/title_dict.pkl', 'rb') as f:
    title_dict = pickle.load(f)

title_df = pd.DataFrame(title_dict.items(), columns=['video_id', 'title'])
title_df.head()


,video_id,title
0,q-wRvsiGYIs,"AMA #19: Collagen vs. Whey Protein, Creatine, ..."
1,ssmwxKPFMFU,Protocols to Improve Vision & Eyesight | Huber...
2,J7yn4tJEmJU,Tools for Overcoming Substance & Behavioral Ad...
3,7MEhDlw1e9k,How to Build Endurance | Huberman Lab Essentials
4,UyneMnERmnI,How to Improve Your Vitality & Heal From Disea...


In [26]:
client = OpenAI(api_key=open_api_key)

max_tokens = 1023 # 8191 is max length for text-embedding-3-large
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")
splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, max_tokens)

# Create lists to store the data
chunk_embeddings = []
title_embeddings = []
ids = []
video_ids = []
chunk_ids = []
chunk_dict = {}


# Iterates through a list of video IDs, where each ID corresponds to a podcast episode transcript.
# Each transcript will be split into chunks and embedded along with its title for semantic search.
documents = list(title_dict.keys())[:30]  # ToDo: process using full dataset 
for idx, video_id in enumerate(documents):

    # Create Title Embeddings
    title_vector = client.embeddings.create(
        input=title_dict[video_id],
        model="text-embedding-3-small"
    )

    # Load document and split into semantic chunks
    try:
        with open(f'data/documents/{video_id}.txt', 'r', encoding='utf-8') as file:
            text_content = file.read()
    except FileNotFoundError:
        print(f"Warning: Document file not found for video ID: {video_id}")
        continue
    # ToDo: validate or rework so chunks are topical sentiments with varying lengths
    chunks = splitter.chunks(text_content) 

    # Create data record of embeddings for each chunk of the transcript
    for chunk in chunks:
        # Create chunks directory if it doesn't exist 
        os.makedirs('data/chunks', exist_ok=True)
        
        # Write chunk to file with video ID and chunk number
        chunk_key = f'_videoid:{video_id}_chunk:{chunks.index(chunk)}'
        chunk_filename = f'data/chunks/{chunk_key}.txt'
        with open(chunk_filename, 'w', encoding='utf-8') as f:
            f.write(chunk)
        
        # Add to chunk to utilitydictionary
        with open(chunk_filename, 'r', encoding='utf-8') as file:
            chunk_content = file.read()
        chunk_content = chunk_content  # .replace('\n', ' ')
        chunk_dict[chunk_key] = chunk_content

        # Create Chunk Embeddings
        chunk_vector = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-small"
        )

        # Append values of this record to column list
        chunk_embeddings.append(chunk_vector.data[0].embedding)
        title_embeddings.append(title_vector.data[0].embedding)
        ids.append(idx)
        video_ids.append(video_id)
        chunk_ids.append(chunk_key)


In [27]:
with open('data/chunk_dict.pkl', 'wb') as f:
    pickle.dump(chunk_dict, f)

In [28]:
# Create DataFrame
df = pd.DataFrame({
    'id': ids,
    'video_id': video_ids, 
    'chunk_id': chunk_ids,
    'content_vector': chunk_embeddings,
    'title_vector': title_embeddings,
})

In [29]:
df.reset_index(inplace=True)
df.rename(columns={'index': 'vector_id'}, inplace=True)
df.head()

,vector_id,id,video_id,chunk_id,content_vector,title_vector
0,0,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:0,"[0.006686230190098286, -0.03623957931995392, -...","[0.027011631056666374, 0.019285723567008972, -..."
1,1,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:1,"[0.02565005235373974, -0.008785320445895195, -...","[0.027011631056666374, 0.019285723567008972, -..."
2,2,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:2,"[0.03873475268483162, -0.03571273759007454, -0...","[0.027011631056666374, 0.019285723567008972, -..."
3,3,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:3,"[0.019842883571982384, -0.019223583862185478, ...","[0.027011631056666374, 0.019285723567008972, -..."
4,4,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:4,"[0.005661938339471817, -0.05714607611298561, -...","[0.027011631056666374, 0.019285723567008972, -..."


In [30]:
# Save DataFrame to CSV
df.to_csv('data/embeddings.csv', index=False)